In [ ]:
# Cell 1: Load Finalized Encoded Dataset & Standardize Features for Clustering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.cluster.hierarchy import dendrogram, linkage

# 1. Load the finalized encoded dataset from Notebook 02
df = pd.read_csv('data/processed/psu_financial_literacy_encoded.csv')

# 2. Exclude derived target columns so clusters form purely from survey responses
target_cols = ['dfl_score', 'dfl_class']
feature_cols = [col for col in df.columns if col not in target_cols]
X_cluster = df[feature_cols]

# 3. Standardize all features (mean = 0, std = 1) for Euclidean distance metrics
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Loaded dataset: {df.shape[0]} respondents, {df.shape[1]} total columns")
print(f"Clustering matrix (X_scaled): {X_scaled.shape[0]} rows x {X_scaled.shape[1]} features")
print(f"Missing values check: {np.isnan(X_scaled).sum()}")

In [ ]:
# Cell 2: Elbow Method & Silhouette Analysis (k = 2 to 8)
k_range = range(2, 9)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

k_summary = pd.DataFrame({
    'k': list(k_range),
    'Inertia (WCSS)': [round(val, 2) for val in inertias],
    'Silhouette Score': [round(val, 4) for val in sil_scores]
})
print("=== K-Means Optimal k Evaluation ===")
print(k_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(k_range), inertias, marker='o', color='#2b6cb0', linewidth=2)
axes[0].set_title('Elbow Method (Within-Cluster Sum of Squares)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(list(k_range), sil_scores, marker='s', color='#2f855a', linewidth=2)
axes[1].set_title('Mean Silhouette Score Across k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Fit K-Means & Hierarchical Clustering + Dendrogram Visualization
# Adjust OPTIMAL_K after inspecting Cell 2's Elbow bend and Silhouette peak
OPTIMAL_K = 3

# 1. Primary Model: K-Means Clustering
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=20)
kmeans_labels = kmeans.fit_predict(X_scaled)

# 2. Validation Benchmark: Hierarchical Clustering (Agglomerative, Ward Linkage)
hierarchical = AgglomerativeClustering(n_clusters=OPTIMAL_K, metric='euclidean', linkage='ward')
hier_labels = hierarchical.fit_predict(X_scaled)

# 3. Plot Hierarchical Clustering Dendrogram (Ward Linkage)
Z = linkage(X_scaled, method='ward', metric='euclidean')

plt.figure(figsize=(12, 5))
dendrogram(
    Z,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90.,
    leaf_font_size=9.,
    show_contracted=True
)
# Draw cut line halfway between the (k-1)th and kth merge distances
cut_height = (Z[-(OPTIMAL_K - 1), 2] + Z[-OPTIMAL_K, 2]) / 2
plt.axhline(y=cut_height, color='#c53030', linestyle='--', linewidth=1.8,
            label=f'Cluster Cut (k = {OPTIMAL_K})')

plt.title('Hierarchical Clustering Dendrogram (Ward Linkage, Truncated to Top 30 Leaves)')
plt.xlabel('Respondent Cluster Size (in parentheses)')
plt.ylabel('Euclidean Distance (Ward)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Quantitative & Visual Comparison (K-Means vs. Hierarchical Clustering)
from scipy.optimize import linear_sum_assignment

# 1. Align Hierarchical cluster IDs to K-Means IDs for clean tables & matching PCA colors
raw_ct = pd.crosstab(kmeans_labels, hier_labels)
row_ind, col_ind = linear_sum_assignment(-raw_ct.values)
label_map = {col_ind[i]: row_ind[i] for i in range(len(row_ind))}
hier_labels_aligned = pd.Series(hier_labels).map(label_map).values

# Update hier_labels so Cell 5/6 also store aligned cluster IDs
hier_labels = hier_labels_aligned

# 2. Internal Cluster Quality Metrics
comparison_df = pd.DataFrame({
    'Algorithm': ['K-Means (Primary)', 'Hierarchical - Ward (Benchmark)'],
    'Silhouette Score (Higher = Better)': [
        round(silhouette_score(X_scaled, kmeans_labels), 4),
        round(silhouette_score(X_scaled, hier_labels), 4)
    ],
    'Calinski-Harabasz Index (Higher = Better)': [
        round(calinski_harabasz_score(X_scaled, kmeans_labels), 2),
        round(calinski_harabasz_score(X_scaled, hier_labels), 2)
    ],
    'Davies-Bouldin Index (Lower = Better)': [
        round(davies_bouldin_score(X_scaled, kmeans_labels), 4),
        round(davies_bouldin_score(X_scaled, hier_labels), 4)
    ]
})
print("=== Internal Cluster Validation Comparison ===")
print(comparison_df.to_string(index=False))

# 3. Cross-Algorithm Agreement
ari = adjusted_rand_score(kmeans_labels, hier_labels)
nmi = normalized_mutual_info_score(kmeans_labels, hier_labels)
print(f"\n=== Cross-Algorithm Cluster Agreement ===")
print(f"Adjusted Rand Index (ARI):            {ari:.4f}")
print(f"Normalized Mutual Information (NMI):  {nmi:.4f}")

# 4. Diagonal-Aligned Contingency Table
print("\n=== Aligned Cluster Assignment Contingency Table ===")
ct = pd.crosstab(
    pd.Series(kmeans_labels, name='K-Means Cluster'),
    pd.Series(hier_labels, name='Hierarchical Cluster (Aligned)')
)
print(ct)

# 5. Side-by-Side 2D PCA Visualization (Matching Colors Across Algorithms)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_explained = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for cluster_id in range(OPTIMAL_K):
    mask_km = (kmeans_labels == cluster_id)
    axes[0].scatter(X_pca[mask_km, 0], X_pca[mask_km, 1], label=f'Cluster {cluster_id}', alpha=0.8, s=55)
axes[0].set_title(f'K-Means Clusters (k = {OPTIMAL_K})')
axes[0].set_xlabel(f'PC1 ({var_explained[0]:.1f}% Variance)')
axes[0].set_ylabel(f'PC2 ({var_explained[1]:.1f}% Variance)')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.4)

for cluster_id in range(OPTIMAL_K):
    mask_hc = (hier_labels == cluster_id)
    axes[1].scatter(X_pca[mask_hc, 0], X_pca[mask_hc, 1], label=f'Cluster {cluster_id}', alpha=0.8, s=55)
axes[1].set_title(f'Hierarchical Ward Clusters (k = {OPTIMAL_K})')
axes[1].set_xlabel(f'PC1 ({var_explained[0]:.1f}% Variance)')
axes[1].set_ylabel(f'PC2 ({var_explained[1]:.1f}% Variance)')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Profile K-Means Clusters by DFL Score, OECD Pass Rate & Top Distinguishing Features
df['kmeans_cluster'] = kmeans_labels
df['hier_cluster'] = hier_labels

# 1. DFL Score & OECD 70/100 Pass Rate per K-Means Cluster
dfl_profile = df.groupby('kmeans_cluster').agg(
    Count=('dfl_score', 'count'),
    Share_Pct=('dfl_score', lambda s: round(len(s) / len(df) * 100, 1)),
    Mean_DFL=('dfl_score', lambda s: round(s.mean(), 2)),
    SD_DFL=('dfl_score', lambda s: round(s.std(), 2)),
    Median_DFL=('dfl_score', lambda s: round(s.median(), 2)),
    OECD_Pass_Rate_Pct=('dfl_class', lambda s: round(s.mean() * 100, 1))
)
print("=== K-Means Cluster Digital Financial Literacy Profile ===")
print(dfl_profile)

# 2. Identify Top Distinguishing Features per Cluster (Standardized Z-Score Deviations)
scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
scaled_df['kmeans_cluster'] = kmeans_labels
cluster_z_means = scaled_df.groupby('kmeans_cluster').mean()

print("\n=== Top Distinguishing Features per Cluster (vs. Cohort Average) ===")
for cid in range(OPTIMAL_K):
    z_series = cluster_z_means.loc[cid].sort_values(ascending=False)
    top_high = z_series.head(5)
    top_low = z_series.tail(5)
    print(f"\n--- Cluster {cid} (n = {(kmeans_labels == cid).sum()}, Mean DFL = {dfl_profile.loc[cid, 'Mean_DFL']}) ---")
    print("Highest above cohort average:")
    for feat, val in top_high.items():
        print(f"  + {feat}: {val:+.2f} SD (raw mean: {df[df['kmeans_cluster'] == cid][feat].mean():.2f})")
    print("Lowest below cohort average:")
    for feat, val in top_low.items():
        print(f"  - {feat}: {val:+.2f} SD (raw mean: {df[df['kmeans_cluster'] == cid][feat].mean():.2f})")

In [ ]:
# Cell 6: Verify & Export Dataset with Cluster Labels
assert df.isnull().sum().sum() == 0, "Unexpected missing values found!"

clustered_output_path = 'data/processed/psu_financial_literacy_clustered.csv'
df.to_csv(clustered_output_path, index=False)
print(f"Saved clustered dataset ({df.shape[0]} rows, {df.shape[1]} columns) to: {clustered_output_path}")